# Clasificación Binaria con Regresión Logística para Predicción de Proveedor de Medicamentos

Proyecto M25-CD | Profesión Científico de Datos v2
Autor: RobertScience

## Objetivo

Desarrollé un modelo de clasificación binaria utilizando Regresión Logística aplicado al dataset clínico de medicamentos.
El objetivo es transformar el problema original y evaluar la capacidad predictiva del modelo mediante métricas de clasificación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, confusion_matrix, classification_report, 
                            roc_curve, roc_auc_score)

In [ ]:
df = pd.read_csv('../data/drugs.csv')
df.head()

In [ ]:
df.info()
df.describe()
df['Drug'].value_counts()

## Transformación de variable objetivo

La variable `Drug` fue transformada en una clasificación binaria.
**DrugC** representa la clase positiva y el resto de medicamentos la clase negativa.

In [ ]:
df['Proveedor'] = df['Drug'].apply(lambda x: 1 if str(x).lower() == 'drugc' else 0)
df[['Drug', 'Proveedor']].head()

In [ ]:
X = df.drop(['Drug', 'Proveedor'], axis=1)
y = df['Proveedor']

encoders = {}
for col in ['Sex', 'BP', 'Cholesterol']:
    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col])
    encoders[col] = encoder

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
solvers = ['liblinear', 'lbfgs', 'saga', 'newton-cg']
resultados = []
modelos = {}

for solver in solvers:
    modelo = LogisticRegression(
        solver=solver, 
        max_iter=1000, 
        random_state=42
    )
    modelo.fit(X_train_scaled, y_train)
    pred = modelo.predict(X_test_scaled)
    
    resultados.append({
        'Solver': solver,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1-score': f1_score(y_test, pred, zero_division=0)
    })
    
    modelos[solver] = modelo

resultados_df = pd.DataFrame(resultados)
resultados_df

In [ ]:
resultados_df.set_index('Solver').plot(
    kind='bar', 
    figsize=(10, 5)
)
plt.title('Comparación de modelos de Regresión Logística')
plt.show()

In [ ]:
mejor_solver = resultados_df.loc[resultados_df['F1-score'].idxmax(), 'Solver']
modelo_final = modelos[mejor_solver]
y_pred_final = modelo_final.predict(X_test_scaled)

print('Modelo seleccionado:', mejor_solver)
print(classification_report(y_test, y_pred_final))

In [ ]:
cm = confusion_matrix(y_test, y_pred_final)

sns.heatmap(cm, annot=True, fmt='d')
plt.title('Matriz de Confusión')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.show()

In [ ]:
y_prob = modelo_final.predict_proba(X_test_scaled)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

print('AUC:', auc)

plt.plot(fpr, tpr, label=f'AUC={auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.legend()
plt.title('Curva ROC')
plt.show()

# Conclusiones finales

El proyecto permitió construir un clasificador binario mediante Regresión Logística.
Se realizó preparación de datos, transformación de variables categóricas, comparación de solvers y evaluación mediante métricas estadísticas.

El modelo seleccionado representa una solución interpretable para escenarios donde se requiere analizar patrones predictivos y explicar resultados.